# 01 — MFG Primer

**Mean-Field Games (MFGs)** model strategic interaction among a continuum of
identical, rational agents.  Each agent optimises against the *aggregate
behaviour* of the population — summarised by the distribution m(t,x).

The equilibrium is a fixed-point: given m*, every agent finds it optimal to
play α*, and α* generates exactly m*.  Mathematically this couples:

* a **backward HJB** PDE for the value function u(t,x), and  
* a **forward Fokker–Planck** PDE for the density m(t,x).

This notebook solves a simple **linear-quadratic (LQ)** example and visualises
the Picard fixed-point iteration.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

import numpy as np
import warnings
warnings.filterwarnings("ignore")

from mfglob.grids import Grid1D, TimeGrid
from mfglob.models.avellaneda_stoikov import AvellanedaStoikovMFG
from mfglob.mfg_solver import MFGSolver


## 1. A simple LQ model — Avellaneda-Stoikov market maker

Agents have inventory `q ∈ [-Q, Q]`, controlled diffusion, quadratic costs.

In [ ]:
model  = AvellanedaStoikovMFG(A=1.0, k=1.5, phi=0.01, gamma=0.1, sigma_mid=0.3)
grid   = Grid1D(-10.0, 10.0, 80)
tgrid  = TimeGrid(0.5, 60)
solver = MFGSolver(model, grid, tgrid, damping=0.5, tol=1e-3, max_iterations=60)
sol    = solver.solve(verbose=False)

print(f"Converged: {sol['converged']}  ({sol['n_iterations']} iterations)")
print(f"Final residual: {sol['convergence_history'][-1]:.4e}")


## 2. Visualise the equilibrium

The density m(t,x) shows the population distribution over inventory; u(t,x) is the optimal value function.

In [ ]:
import numpy as np

m      = sol['density']          # (n_t+1, n_x)
u      = sol['value_function']   # (n_t+1, n_x)
alpha  = sol['optimal_control']  # (n_t+1, n_x)
x      = grid.points

# Terminal statistics
m_T = m[-1]
mass = np.trapezoid(m_T, x)
mean = np.trapezoid(x * m_T, x)
std  = np.sqrt(np.trapezoid((x - mean)**2 * m_T, x))

print("Terminal density m(T, x):")
print(f"  mass  = {mass:.6f}  (should be ≈ 1)")
print(f"  mean  = {mean:.4f}")
print(f"  std   = {std:.4f}")

print("\nValue function u(0, ·):")
print(f"  min = {u[0].min():.4f},  max = {u[0].max():.4f}")


## 3. Picard iteration convergence

The residual `‖m^{n+1} - m^n‖` should decrease geometrically.

In [ ]:
history = sol['convergence_history']
print("Iteration  |  residual")
print("-" * 28)
for i, r in enumerate(history):
    bar = "█" * int(40 * r / max(history))
    print(f"  {i+1:3d}    |  {r:.4e}  {bar}")


## 4. The fixed-point equations

Given damping λ ∈ (0,1], the Lasry-Lions algorithm converges when the
monotonicity condition holds:

```
∫ [f(x, m₁) − f(x, m₂)] (m₁ − m₂) dx  ≥  0   ∀ m₁, m₂
```

For congestion-type coupling `f(x,m) = φ·m(x)` this is satisfied with
constant φ > 0, guaranteeing a unique MFG equilibrium.

**Key result (Lasry & Lions 2007):** Under the monotonicity condition and
standard regularity, there exists a unique classical solution (u, m) to the
coupled HJB–FP system.


In [ ]:
# Verify mass conservation across time
masses = [float(np.trapezoid(m[t], x)) for t in range(len(m))]
print(f"Mass at t=0: {masses[0]:.8f}")
print(f"Mass at t=T: {masses[-1]:.8f}")
print(f"Max deviation: {max(abs(mm - 1.0) for mm in masses):.2e}")
